In [1]:
import requests
import pandas as pd
import json
from datetime import datetime
import re
import os
from configparser import ConfigParser
import TELTools as TT

c:\Users\214225\AppData\Local\Programs\Python\Python37\lib\site-packages\snowflake\connector\options.py:109: UserWarning: You have an incompatible version of 'pyarrow' installed (5.0.0), please install a version that adheres to: 'pyarrow<10.1.0,>=10.0.1; extra == "pandas"'
  "pyarrow", installed_pyarrow_version, pandas_pyarrow_extra
Failed to import ArrowResult. No Apache Arrow result set format can be used. ImportError: DLL load failed: The specified procedure could not be found.


In [3]:
# get api key from azure kv
SAP_API_KEY = 'SAP-API-KEY'
API_KEY = TT.getAzureSecret(SAP_API_KEY).value

In [8]:
# get config file
SAP_cdsv_config = '\\telusdwappprod\\f$\\Shares\\Configuration\\SAP_cdsv.ini'

config = ConfigParser()
config.read(SAP_cdsv_config)

cds_views = []
cds_views.append(['ZCDV_T_CMTU024','ACDOCA'])
# for section in config.sections():
#     base_table_name = config.get(section,'base_table_name')
#     cds_views.append([section,base_table_name])


In [35]:
# api call

count_api_list = []

# prod environment
url = 'http://cdsv-odata-3scale-apicast-production.usr.ocp.tel.com'
client = 'client100'
appID = '0b9a4baf'

for cds in cds_views:
    cds_view = cds[0]
    base_table_name = cds[1]
    cnt = 0
    # print(cds_view)
    SERVICE_URL = \
    url+\
"/v1/odata/"\
+client+\
"/sap/opu/odata/sap/"\
+cds_view+\
"_CDS/"\
+cds_view+\
"?\
app_id="\
+appID+\
"&app_key="\
+API_KEY+\
"&sap_user=IF_API_SNOWF\
&$format=json\
&$filter=Update_Date+eq+''\
&$inlinecount=allpages\
&$orderby=\
&$select=\
&$skip=\
&$top=1000"

    print(SERVICE_URL)
    try:
        response = requests.get(SERVICE_URL, headers = {"Prefer": "odata.maxpagesize=500","Prefer": "odata.track-changes"})

        # load JSON response Python dictionary object
        status_code = response.status_code
        # print(cds_view,status_code) 
        init_json = json.loads(response.content)
        if status_code == 200:
            cnt = init_json['d']['__count']
        message = f'{cds_view},{base_table_name},{status_code},{cnt},success'
        print(message)
        count_api_list.append(message)

    except Exception as e:
        print(f'{cds_view} failed: {e}')
        message = f'{cds_view},{base_table_name},{status_code},{cnt},{e}'
        count_api_list.append(message)
        continue

# print(count_api_list)
    

http://cdsv-odata-3scale-apicast-production.usr.ocp.tel.com/v1/odata/client100/sap/opu/odata/sap/ZCDV_T_CMTU024_CDS/ZCDV_T_CMTU024?app_id=0b9a4baf&app_key=7b3906eb49c28884a544a201b0fe2a73&sap_user=IF_API_SNOWF&$format=json&$filter=Update_Date+eq+''&$inlinecount=allpages&$orderby=&$select=&$skip=&$top=1000
ZCDV_T_CMTU024,ACDOCA,200,0,success


In [36]:
print(init_json)
out_file = 'c:\\temp\\output.txt'
with open(out_file,'w') as out:
    json.dump(init_json,out) 

{'d': {'__count': '0', 'results': []}}


In [15]:
# output results to csv
file_name = 'api_counts.csv'
df = pd.DataFrame(data=count_api_list)
df.to_csv(file_name, sep='|', encoding='utf-8',index=False,header=False)
    